# KAN-Vim trên Google Colab (T4)
**Trước khi chạy:**
1. Upload `kan-vim.zip` lên Google Drive (để ở đâu cũng được, cell 1 tự tìm).
2. *Thời gian chạy -> Thay đổi loại thời gian chạy -> T4 GPU*.
3. Chạy lần lượt từng cell từ trên xuống. Mỗi lần mở lại / khởi động lại phiên: chạy lại từ cell 1.

## Cell 1 — Chuẩn bị: GPU, Drive, giải nén code, tải checkpoint, chạy test
Mong đợi: `cuda ok: True`, danh sách `kanvim` có `vim_pure.py`, `ckpt ok`, `41 passed`.

In [ ]:
import os, sys, glob, shutil, subprocess, torch
print('torch', torch.__version__, '| python', sys.version.split()[0], '| cuda ok:', torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
RES = f'{DRIVE}/kan-vim-results'
os.makedirs(RES, exist_ok=True)

os.chdir('/content')
zips = sorted(glob.glob(f'{DRIVE}/**/kan-vim*.zip', recursive=True), key=os.path.getmtime)
assert zips, 'Không tìm thấy kan-vim.zip trên Drive -> upload lên rồi chạy lại cell này'
print('zip:', zips[-1])
shutil.rmtree('/content/kan-vim', ignore_errors=True)
subprocess.run(['unzip', '-q', '-o', zips[-1], '-d', '/content'], check=True)
os.chdir('/content/kan-vim')
sys.path.insert(0, '/content/kan-vim')
print('kanvim:', sorted(os.listdir('kanvim')))

from huggingface_hub import hf_hub_download
p = hf_hub_download('hustvl/Vim-tiny-midclstok', 'vim_t_midclstok_76p1acc.pth')
shutil.copy(p, '/content/kan-vim/vim_t.pth')
print('ckpt ok', os.path.getsize('/content/kan-vim/vim_t.pth') // 2**20, 'MB')

out = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], capture_output=True, text=True)
print((out.stdout + out.stderr).strip().splitlines()[-1])

## Cell 2 — Kiểm tra bản Vim thuần PyTorch với ảnh chó Samoyed
Mong đợi: in ra **ĐÚNG** (class 258 nằm trong top-5).

In [ ]:
import os, sys, urllib.request, torch
os.chdir('/content/kan-vim'); sys.path.insert(0, '/content/kan-vim')
from PIL import Image
from torchvision import transforms as T
from kanvim.vim_pure import load_vim_pure

m = load_vim_pure('tiny', '/content/kan-vim/vim_t.pth', num_classes=1000).cuda().eval()
urllib.request.urlretrieve('https://github.com/pytorch/hub/raw/master/images/dog.jpg', '/content/dog.jpg')
tf = T.Compose([T.Resize(256, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224), T.ToTensor(),
                T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
x = tf(Image.open('/content/dog.jpg').convert('RGB')).unsqueeze(0).cuda()
with torch.no_grad():
    prob = m(x).softmax(-1)[0]
top = prob.topk(5)
print('top5 class ids:', top.indices.tolist(), '| probs:', [round(v, 3) for v in top.values.tolist()])
print('ĐÚNG' if 258 in top.indices.tolist() else 'SAI -> gửi output này cho Claude')
del m; torch.cuda.empty_cache()

## Cell 3 — Đo tốc độ trên T4 (khoảng 5-10 phút, lần đầu tự tải CIFAR-100)
Ghi lại các dòng `s/step -> min/epoch` và `saved ...`, **gửi cho Claude** trước khi chạy tiếp.

In [ ]:
%cd /content/kan-vim
!python train.py --dataset cifar100 --data_root /content/data --backbone tiny --ckpt vim_t.pth --reducer identity --ratio 1.0 --epochs 1 --max_steps 30 --batch 32 --grad_ckpt --out /content/results_timing
!python train.py --dataset cifar100 --data_root /content/data --backbone tiny --ckpt vim_t.pth --reducer kan_prune --ratio 0.5 --epochs 1 --max_steps 30 --batch 32 --grad_ckpt --out /content/results_timing

---
# Các cell dưới: CHỈ chạy sau khi Claude chốt số epoch

## Cell 4 — Baseline (100% token), lưu checkpoint làm teacher

In [ ]:
%cd /content/kan-vim
EPOCHS = 5   # Claude sẽ chốt con số này
!python train.py --dataset cifar100 --data_root /content/data --backbone tiny --ckpt vim_t.pth --reducer identity --ratio 1.0 --epochs {EPOCHS} --batch 32 --grad_ckpt --seed 0 --out {RES} --save_ckpt {RES}/teacher_cifar100_tiny.pt

## Cell 5 — Chạy một đợt nhỏ (kết quả lưu trên Drive, không mất khi Colab ngắt)

In [ ]:
%cd /content/kan-vim
EXTRA = f"--data_root /content/data --batch 32 --grad_ckpt --out {RES} --teacher_ckpt {RES}/teacher_cifar100_tiny.pt"
!python scripts/sweep.py --dataset cifar100 --backbone tiny --ckpt vim_t.pth --epochs {EPOCHS} --seeds 0 --no_baseline --reducers kan_prune mlp_prune --ratios 0.5 --extra "{EXTRA}" > batch1.sh
!cat batch1.sh
!bash batch1.sh

## Cell 6 — Bảng kết quả

In [ ]:
%cd /content/kan-vim
!python scripts/aggregate.py {RES}